# Event Registration Portal

A console portal where societies register their events, the Student Centre approves them, and students can see all upcoming events and book tickets.

Data is saved in three text files:
users.txt: roll number, name, password
events.txt: event id, title, society, date, venue, category, seats, status, posted by, attendees
tickets.txt: ticket id, event id, roll number

In [1]:
# EVENT REGISTRATION PORTAL
# FILE HANDLING

from datetime import datetime

USER_FILE = "users.txt"
EVENT_FILE = "events.txt"
TICKET_FILE = "tickets.txt"

# password for the Student Centre (admin) login
ADMIN_PASSWORD = "admin123"

# icons for each type of event
CATEGORY_ICONS = {
    "tech": "💻",
    "sports": "⚽",
    "music": "🎵",
    "art": "🎨",
    "seminar": "🎤",
    "social": "🎉",
    "other": "📌",
}


def load_users():
    """Reads users from users.txt into a list of dictionaries."""
    users = []
    try:
        with open(USER_FILE, "r") as user_file:
            for line in user_file:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("|")
                if len(parts) == 3:
                    roll, name, password = parts
                    users.append(
                        {
                            "roll_number": roll,
                            "name": name,
                            "password": password,
                        }
                    )
    except FileNotFoundError:
        pass  # Starts empty if file does not exist yet
    except Exception as e:
        print(f"[Warning] Error loading users: {e}")
    return users


def save_users(users):
    """Saves current users list to users.txt."""
    try:
        with open(USER_FILE, "w") as user_file:
            for u in users:
                user_file.write(f"{u['roll_number']}|{u['name']}|{u['password']}\n")
        return True
    except PermissionError:
        print("[Error] No permission to write users.txt")
        return False
    except Exception as e:
        print(f"[Error] Error saving users: {e}")
        return False


def load_events():
    """Reads events from events.txt into a list of dictionaries."""
    events = []
    try:
        with open(EVENT_FILE, "r") as event_file:
            for line in event_file:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("|")
                if len(parts) == 10:
                    e_id, title, society, date, venue, category, seats, status, posted_by, attendees_str = parts
                    events.append(
                        {
                            "event_id": e_id,
                            "title": title,
                            "society": society,
                            "date": date,
                            "venue": venue,
                            "category": category,
                            "seats": int(seats),
                            "status": status,
                            "posted_by": posted_by,
                            "attendees": [
                                a.strip()
                                for a in attendees_str.split(",")
                                if a.strip()
                            ],
                        }
                    )
    except FileNotFoundError:
        pass  
    except ValueError:
        print("[Warning] Some event has wrong seats number in events.txt")
    except Exception as e:
        print(f"[Warning] Error loading events: {e}")
    return events


def save_events(events):
    """Saves current events list to events.txt."""
    try:
        with open(EVENT_FILE, "w") as event_file:
            for ev in events:
                attendees = ",".join(ev["attendees"])
                event_file.write(f"{ev['event_id']}|{ev['title']}|{ev['society']}|{ev['date']}|{ev['venue']}|{ev['category']}|{ev['seats']}|{ev['status']}|{ev['posted_by']}|{attendees}\n")
        return True
    except PermissionError:
        print("[Error] No permission to write events.txt")
        return False
    except Exception as e:
        print(f"[Error] Error saving events: {e}")
        return False


def save_ticket(ticket_id, event_id, roll):
    """Adds one new ticket at the end of tickets.txt (append mode)."""
    try:
        with open(TICKET_FILE, "a") as ticket_file:
            ticket_file.write(f"{ticket_id}|{event_id}|{roll}\n")
        return True
    except Exception as e:
        print(f"[Error] Error saving ticket: {e}")
        return False


def load_tickets():
    """Reads all tickets from tickets.txt into a list of dictionaries."""
    tickets = []
    try:
        with open(TICKET_FILE, "r") as ticket_file:
            for line in ticket_file:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("|")
                if len(parts) == 3:
                    tickets.append(
                        {
                            "ticket_id": parts[0],
                            "event_id": parts[1],
                            "roll_number": parts[2],
                        }
                    )
    except FileNotFoundError:
        pass  # no tickets booked yet
    except Exception as e:
        print(f"[Warning] Error loading tickets: {e}")
    return tickets


## 2. Accounts
Students register and log in. The Student Centre has its own login (password:admin123) to approve events.

In [2]:
# ACCOUNT FUNCTIONS (STUDENT + STUDENT CENTRE)

def register_user(users):
    """Registers a new student account."""
    print("\n--- Register Student Account ---")
    roll = input("Enter Roll Number (e.g. 25L-5150): ").strip()
    if roll == "":
        print("❌ Roll Number cannot be empty!")
        return users, None
    for u in users:
        if u["roll_number"].lower() == roll.lower():
            print("❌ Error: A student with this Roll Number already exists!")
            return users, None
    name = input("Enter Full Name: ").strip()
    password = input("Enter Password: ").strip()
    if name == "" or password == "":
        print("❌ Name and Password cannot be empty!")
        return users, None
    new_user = {
        "roll_number": roll,
        "name": name,
        "password": password,
    }
    users.append(new_user)
    if save_users(users):
        print(f"✅ Account registered successfully for {name}!")
        return users, new_user
    return users, None


def user_login(users):
    """Logs in an existing student."""
    print("\n--- Student Login ---")
    roll = input("Enter Roll Number: ").strip()
    password = input("Enter Password: ").strip()
    for u in users:
        if u["roll_number"].lower() == roll.lower() and u["password"] == password:
            print(f"\n🎉 Login successful! Welcome {u['name']}.")
            return u
    print("❌ Invalid Roll Number or Password.")
    return None


def admin_login():
    """Logs in the Student Centre so they can approve events."""
    print("\n--- Student Centre Login ---")
    password = input("Enter Student Centre Password: ").strip()
    if password == ADMIN_PASSWORD:
        print("\n🏛️ Welcome Student Centre!")
        return True
    print("❌ Wrong password.")
    return False


## 3. Events
add_record(): a society registers a new event, its status is Pending

approve_events(): Student Centre approves or rejects pending events

view_upcoming_events(): shows approved upcoming events as an icon list with their societies

search_record(): search events by title, society, venue or category

The date is checked with datetime, a wrong date gives ValueError which we catch.

In [3]:
# EVENT FUNCTIONS

def get_icon(category):
    """Returns the icon of a category, or a pin if not found."""
    if category in CATEGORY_ICONS:
        return CATEGORY_ICONS[category]
    return CATEGORY_ICONS["other"]


def is_upcoming(date_text):
    """Checks if an event date is today or later."""
    try:
        event_date = datetime.strptime(date_text, "%d-%m-%Y")
        today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        return event_date >= today
    except ValueError:
        return False


def add_record(events, current_user):
    """Society registers a new event. It waits for Student Centre approval."""
    print("\n--- Register New Society Event ---")
    society = input("Enter Society Name (e.g. ACM, Dramatics): ").strip()
    title = input("Enter Event Title: ").strip()
    if society == "" or title == "":
        print("❌ Society name and title cannot be empty!")
        return events


    while True:
        date = input("Enter Event Date (DD-MM-YYYY): ").strip()
        try:
            datetime.strptime(date, "%d-%m-%Y")
            if is_upcoming(date):
                break
            print("❌ Date has already passed! Enter a future date.")
        except ValueError:
            print("❌ Invalid date! Use format like 25-12-2026.")

    venue = input("Enter Venue: ").strip()

    print("Categories: tech, sports, music, art, seminar, social, other")
    category = input("Enter Category: ").strip().lower()
    if category not in CATEGORY_ICONS:
        print("⚠️ Unknown category, saving it as 'other'.")
        category = "other"

    while True:
        try:
            seats = int(input("Enter Total Seats / Tickets (e.g. 50): ").strip())
            if seats > 0:
                break
            print("Please enter a positive number.")
        except ValueError:
            print("❌ Invalid input! Enter a valid integer.")

    e_id = f"EVT-{len(events) + 1:03d}"
    new_event = {
        "event_id": e_id,
        "title": title,
        "society": society,
        "date": date,
        "venue": venue,
        "category": category,
        "seats": seats,
        "status": "Pending",
        "posted_by": current_user["roll_number"],
        "attendees": [],
    }
    events.append(new_event)
    if save_events(events):
        print(f"✅ Event '{title}' sent to Student Centre for approval! ID: {e_id}")
    return events


def approve_events(events):
    """Student Centre approves or rejects pending events."""
    print("\n--- Pending Event Requests ---")
    pending = []
    for ev in events:
        if ev["status"] == "Pending":
            pending.append(ev)

    if not pending:
        print("📭 No pending events right now.")
        return events

    for ev in pending:
        print(f"\n{get_icon(ev['category'])} [{ev['event_id']}] {ev['title']}")
        print(f"Society: {ev['society']} | Date: {ev['date']} | Venue: {ev['venue']}")
        print(f"Seats: {ev['seats']} | Requested by: {ev['posted_by']}")
        choice = input("Approve (A) / Reject (R) / Skip (S): ").strip().upper()
        if choice == "A":
            ev["status"] = "Approved"
            print("✅ Approved!")
        elif choice == "R":
            ev["status"] = "Rejected"
            print("🚫 Rejected!")
        else:
            print("⏭️ Skipped.")

    save_events(events)
    return events


def view_upcoming_events(events):
    """Shows all approved upcoming events with their icons and societies."""
    print("\n==========================================")
    print("          📅 UPCOMING EVENTS 📅           ")
    print("==========================================")
    count = 0
    for ev in events:
        if ev["status"] == "Approved" and is_upcoming(ev["date"]):
            count = count + 1
            seats_left = ev["seats"] - len(ev["attendees"])
            print(f"\n{get_icon(ev['category'])}  {ev['title']}  [{ev['event_id']}]")
            print(f"   Society : {ev['society']}")
            print(f"   Date    : {ev['date']}")
            print(f"   Venue   : {ev['venue']}")
            if seats_left > 0:
                print(f"   Tickets : {seats_left} left out of {ev['seats']}")
            else:
                print("   Tickets : SOLD OUT ❌")
            print("-" * 40)
    if count == 0:
        print("📭 No upcoming events yet. Check back later!")
    else:
        print("👉 To get a ticket, choose 'Enroll in an Event' and enter the Event ID.")


def search_record(events):
    """Searches approved events by title, society, venue or category."""
    print("\n--- Search Events ---")
    query = input("Enter keyword to search: ").strip().lower()
    found = []
    for ev in events:
        if ev["status"] != "Approved":
            continue
        in_title = query in ev["title"].lower()
        in_society = query in ev["society"].lower()
        in_venue = query in ev["venue"].lower()
        in_category = query in ev["category"].lower()
        if in_title or in_society or in_venue or in_category:
            found.append(ev)
    if not found:
        print("🔍 No matching events found.")
        return
    print(f"\nFound {len(found)} event(s):")
    for ev in found:
        print(f"\n{get_icon(ev['category'])} ID: {ev['event_id']} | {ev['title']} | Society: {ev['society']}")
        print(f"Date: {ev['date']} | Venue: {ev['venue']}")
        print(f"Tickets booked: {len(ev['attendees'])}/{ev['seats']}")
        print("-" * 40)


## 4. Tickets
enroll_event() books a ticket if the event is approved, upcoming, and has seats left. Tickets are appended to tickets.txt.

In [4]:
# TICKET FUNCTIONS

def enroll_event(events, current_user):
    """Books a ticket for the student in an approved event."""
    print("\n--- Enroll in an Event ---")
    e_id = input("Enter Event ID (e.g. EVT-001): ").strip().upper()
    for ev in events:
        if ev["event_id"].upper() == e_id:
            if ev["status"] != "Approved":
                print("⏳ This event is not approved by Student Centre yet!")
                return events
            if not is_upcoming(ev["date"]):
                print("❌ This event has already happened.")
                return events
            if current_user["roll_number"] in ev["attendees"]:
                print("⚠️ You already have a ticket for this event!")
                return events
            if len(ev["attendees"]) >= ev["seats"]:
                print("❌ Sorry, tickets are sold out!")
                return events

            ev["attendees"].append(current_user["roll_number"])
            save_events(events)

            # ticket number = event id + seat number
            ticket_id = f"TKT-{ev['event_id']}-{len(ev['attendees']):03d}"
            save_ticket(ticket_id, ev["event_id"], current_user["roll_number"])

            print("\n🎟️ ================================ 🎟️")
            print("            YOUR EVENT TICKET          ")
            print("🎟️ ================================ 🎟️")
            print(f"Ticket No : {ticket_id}")
            print(f"Name      : {current_user['name']} ({current_user['roll_number']})")
            print(f"Event     : {get_icon(ev['category'])} {ev['title']}")
            print(f"Society   : {ev['society']}")
            print(f"Date      : {ev['date']}")
            print(f"Venue     : {ev['venue']}")
            print("Show this ticket at the entry. Enjoy! 😊")
            return events
    print("❌ Event ID not found.")
    return events


def view_my_tickets(events, current_user):
    """Shows all tickets booked by the logged in student."""
    print("\n--- My Tickets ---")
    tickets = load_tickets()
    count = 0
    for t in tickets:
        if t["roll_number"] == current_user["roll_number"]:
            # find the event of this ticket
            for ev in events:
                if ev["event_id"] == t["event_id"]:
                    count = count + 1
                    print(f"🎟️ {t['ticket_id']} -> {get_icon(ev['category'])} {ev['title']} on {ev['date']} at {ev['venue']}")
    if count == 0:
        print("You have not booked any tickets yet.")


## 5. Summary Report
Counts approved, pending and rejected events, total tickets, and events per society.

In [5]:
# REPORT FUNCTION

def generate_report(events, users):
    """Generates portal statistics report."""
    print("\n==========================================")
    print("      PORTAL SUMMARY & ANALYTICS REPORT    ")
    print("==========================================")
    print(f"Total Registered Students: {len(users)}")
    print(f"Total Events Submitted: {len(events)}")

    approved = 0
    pending = 0
    rejected = 0
    total_tickets = 0
    for ev in events:
        if ev["status"] == "Approved":
            approved = approved + 1
        elif ev["status"] == "Pending":
            pending = pending + 1
        else:
            rejected = rejected + 1
        total_tickets = total_tickets + len(ev["attendees"])

    print(f"Approved Events: {approved}")
    print(f"Pending Approval: {pending}")
    print(f"Rejected Events: {rejected}")
    print(f"Total Tickets Booked: {total_tickets}")
    print("==========================================")

    # count events of every society
    society_count = {}
    for ev in events:
        if ev["status"] == "Approved":
            if ev["society"] in society_count:
                society_count[ev["society"]] = society_count[ev["society"]] + 1
            else:
                society_count[ev["society"]] = 1
    if society_count:
        print("\n--- EVENTS PER SOCIETY ---")
        for society in society_count:
            print(f"🏷️ {society}: {society_count[society]} event(s)")

    if events:
        print("\n--- ALL EVENT DETAILS ---")
        for ev in events:
            print(f"{get_icon(ev['category'])} [{ev['event_id']}] {ev['title']} ({ev['society']}) - {ev['status']}")
            print(f"Date: {ev['date']} | Tickets: {len(ev['attendees'])}/{ev['seats']}")
            print("-" * 40)


## 6. Main Menu
The while True loop keeps the portal running until the user chooses Exit. There are three menus: guest, student and Student Centre.

In [ ]:
# ==========================================
# MAIN INTERACTIVE MENU LOOP
# ==========================================
def main():
    users = load_users()
    events = load_events()
    current_user = None
    is_admin = False
    while True:
        print("\n==========================================")
        print("        EVENT REGISTRATION PORTAL         ")
        print("==========================================")
        if is_admin:
            print("Logged in: Student Centre 🏛️")
            print("1. Approve / Reject Events")
            print("2. View Upcoming Events")
            print("3. View Summary Report")
            print("4. Logout")
            choice = input("\nSelect option (1-4): ").strip()
            if choice == "1":
                events = approve_events(events)
            elif choice == "2":
                view_upcoming_events(events)
            elif choice == "3":
                generate_report(events, users)
            elif choice == "4":
                print("\nStudent Centre logged out.")
                is_admin = False
            else:
                print("❌ Invalid selection! Choose 1, 2, 3, or 4.")
        elif not current_user:
            print("1. Register Account")
            print("2. Student Login")
            print("3. Student Centre Login")
            print("4. View Upcoming Events (Guest)")
            print("5. Exit")
            choice = input("\nSelect option (1-5): ").strip()
            if choice == "1":
                users, current_user = register_user(users)
            elif choice == "2":
                current_user = user_login(users)
            elif choice == "3":
                is_admin = admin_login()
            elif choice == "4":
                view_upcoming_events(events)
            elif choice == "5":
                print("\nExiting portal. All records saved. Goodbye👋😊!")
                break
            else:
                print("❌ Invalid selection! Choose 1, 2, 3, 4 or 5.")
        else:
            print(f"Logged in: {current_user['name']} ({current_user['roll_number']})")
            print("1. Register a Society Event")
            print("2. View Upcoming Events")
            print("3. Search Events")
            print("4. Enroll in an Event (Get Ticket)")
            print("5. View My Tickets")
            print("6. View Summary Report")
            print("7. Logout")
            print("8. Exit")
            choice = input("\nSelect option (1-8): ").strip()
            if choice == "1":
                events = add_record(events, current_user)
            elif choice == "2":
                view_upcoming_events(events)
            elif choice == "3":
                search_record(events)
            elif choice == "4":
                events = enroll_event(events, current_user)
            elif choice == "5":
                view_my_tickets(events, current_user)
            elif choice == "6":
                generate_report(events, users)
            elif choice == "7":
                print(f"\nLogged out {current_user['name']}.")
                current_user = None
            elif choice == "8":
                print("\nExiting portal. All records saved. Goodbye👋😊!")
                break
            else:
                print("❌ Invalid selection! Choose between 1 and 8.")


# Run 
if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nPortal stopped by user.")
    finally:
        print("Thank you for using the Event Registration Portal!")



        EVENT REGISTRATION PORTAL         
1. Register Account
2. Student Login
3. Student Centre Login
4. View Upcoming Events (Guest)
5. Exit



Select option (1-5):  1



--- Register Student Account ---


Enter Roll Number (e.g. 25L-5150):  25L-5252
Enter Full Name:  fatima
Enter Password:  1234


✅ Account registered successfully for fatima!

        EVENT REGISTRATION PORTAL         
Logged in: fatima (25L-5252)
1. Register a Society Event
2. View Upcoming Events
3. Search Events
4. Enroll in an Event (Get Ticket)
5. View My Tickets
6. View Summary Report
7. Logout
8. Exit


## 7. Checking the Saved Files
to see the data saved in the text files.

In [ ]:
with open("events.txt", "r") as event_file:
    print(event_file.read())

In [ ]:
with open("tickets.txt", "r") as ticket_file:
    print(ticket_file.read())